# Feature Engineering

On part du dataset nettoyé (`cs-training-clean.csv`) produit par `preprocess.py` et on crée de nouvelles variables pour aider le modèle à mieux détecter les profils risqués.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load import load_raw

df_raw = load_raw()
df_clean = pd.read_csv("../data/processed/cs-training-clean.csv")
df_features = df_clean.copy()

print(f"Dataset brut    : {df_raw.shape}")
print(f"Dataset nettoyé : {df_features.shape}")


## 1. Indicateurs de données manquantes

Le fait de ne pas avoir déclaré son revenu est en soi un signal de risque. On crée une colonne binaire (0/1) pour que le modèle sache que la valeur a été imputée, pas déclarée par le client.

In [ ]:
df_features["MonthlyIncome_is_missing"] = df_raw["MonthlyIncome"].isna().astype(int)
df_features["NumberOfDependents_is_missing"] = df_raw["NumberOfDependents"].isna().astype(int)

print(df_features["MonthlyIncome_is_missing"].value_counts())


## 2. Agrégation des retards de paiement

On a 3 colonnes de retard (30-59j, 60-89j, 90j+). On les combine en :
- **TotalPastDue** : somme de tous les retards
- **WeightedPastDue** : somme pondérée (×1, ×2, ×3 selon la gravité)
- **HasAnyPastDue** : binaire, a eu au moins un retard


In [ ]:
df_features["TotalPastDue"] = (
    df_features["NumberOfTime30-59DaysPastDueNotWorse"] +
    df_features["NumberOfTime60-89DaysPastDueNotWorse"] +
    df_features["NumberOfTimes90DaysLate"]
)

df_features["WeightedPastDue"] = (
    df_features["NumberOfTime30-59DaysPastDueNotWorse"] * 1 +
    df_features["NumberOfTime60-89DaysPastDueNotWorse"] * 2 +
    df_features["NumberOfTimes90DaysLate"] * 3
)

df_features["HasAnyPastDue"] = (df_features["TotalPastDue"] > 0).astype(int)

## 3. Variables financières dérivées

- **IncomePerPerson** : revenu divisé par la taille du foyer. Un salaire de 3000€ n'a pas le même poids pour un célibataire que pour une famille de 6.
- **MonthlyDebt** : dette mensuelle absolue (DebtRatio × MonthlyIncome)
- **RealEstateToCreditLinesRatio** : proportion de crédits immobiliers par rapport au total des lignes de crédit


In [ ]:
df_features["IncomePerPerson"] = df_features["MonthlyIncome"] / (df_features["NumberOfDependents"] + 1)

df_features["MonthlyDebt"] = df_features["DebtRatio"] * df_features["MonthlyIncome"]

df_features["RealEstateToCreditLinesRatio"] = np.where(
    df_features["NumberOfOpenCreditLinesAndLoans"] == 0,
    0,
    df_features["NumberRealEstateLoansOrLines"] / df_features["NumberOfOpenCreditLinesAndLoans"]
)

## 4. Validation : corrélation avec la cible

On mesure la corrélation de Pearson de chaque nouvelle variable avec `SeriousDlqin2yrs` pour vérifier quelles features apportent réellement un signal.

In [ ]:
new_features = [
    "SeriousDlqin2yrs",
    "WeightedPastDue", "TotalPastDue", "HasAnyPastDue",
    "MonthlyIncome_is_missing", "IncomePerPerson",
    "MonthlyDebt", "RealEstateToCreditLinesRatio", "NumberOfDependents_is_missing"
]

corr = df_features[new_features].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True)
plt.title("Matrice de corrélation des nouvelles features")
plt.tight_layout()
plt.show()

## Conclusion

### Features retenues

- **WeightedPastDue** (0.394): Le signal le plus fort du dataset. Résume les 3 colonnes de retard en un seul score pondéré par gravité.
- **HasAnyPastDue** (0.314): Indicateur binaire simple : le client a-t-il eu au moins un retard ?
- **MonthlyIncome_is_missing** (-0.021): Faible en linéaire, mais potentiellement utile pour les modèles à base d'arbres.
- **IncomePerPerson** (-0.024): Capture le "reste à vivre" réel du foyer.

### Features écartées

- **TotalPastDue** (0.390): Redondant avec WeightedPastDue (corrélation mutuelle = 0.939). On garde WeightedPastDue qui pondère par gravité.
- **MonthlyDebt** (-0.004): Aucun signal.
- **RealEstateToCreditLinesRatio** (-0.010): Aucun signal.
- **NumberOfDependents_is_missing** (-0.014): Aucun signal, et fortement lié à MonthlyIncome_is_missing (0.330).
